# Converse API のサンプル

Amazon Bedrock の Converse API を使用した基本的なサンプル集です。
段階的に機能を追加しながら、Converse API の使い方を学びます。

## 1. シンプルな Converse API の呼び出し（inferenceConfig・system なし）

最もシンプルな形で Converse API を呼び出すサンプルです。  
inferenceConfig も system メッセージも使用せず、ユーザーメッセージのみを送信します。

In [ ]:
import boto3

# Bedrock Runtime クライアントの作成
bedrock_client = boto3.client('bedrock-runtime')

# 使用するモデルの指定
model_id = "amazon.nova-lite-v1:0"

# ユーザーメッセージの作成（日本の祭りについて詳しく聞く）
messages = [
    {
        "role": "user",
        "content": [{"text": "日本の三大祭り（祇園祭、天神祭、神田祭）について、それぞれの歴史、開催時期、見どころ、特徴的な行事を詳しく教えてください。"}]
    }
]

# Converse API の呼び出し（inferenceConfig も system も指定しない）
response = bedrock_client.converse(
    modelId=model_id,
    messages=messages
)

# レスポンスの表示
output_message = response['output']['message']
print(f"Role: {output_message['role']}")
print(f"\n{output_message['content'][0]['text']}")

# トークン使用量の表示
usage = response['usage']
print(f"\n--- トークン使用量 ---")
print(f"入力トークン数: {usage['inputTokens']}")
print(f"出力トークン数: {usage['outputTokens']}")
print(f"合計トークン数: {usage['totalTokens']}")

## 2. system プロンプトを使用した Converse API の呼び出し

system プロンプトを使用して、モデルの振る舞いを制御するサンプルです。  
日本の歴史に詳しいガイドとして回答するよう指示しています。

In [ ]:
import boto3

# Bedrock Runtime クライアントの作成
bedrock_client = boto3.client('bedrock-runtime')

# 使用するモデルの指定
model_id = "amazon.nova-lite-v1:0"

# system プロンプトの設定（日本文化の専門家として振る舞う）
system_prompts = [
    {
        "text": "あなたは日本の伝統文化と歴史に精通した専門家です。質問に対して、歴史的背景、文化的意義、現代との関連性を含めて、豊富な具体例とともに詳しく解説してください。回答は見出しや箇条書きを使って構造的に整理してください。"
    }
]

# ユーザーメッセージの作成
messages = [
    {
        "role": "user",
        "content": [{"text": "日本の四季と伝統行事の関係について教えてください。春夏秋冬それぞれの代表的な行事や風習、食文化、自然との関わりについて詳しく説明してください。"}]
    }
]

# Converse API の呼び出し（system プロンプト付き、inferenceConfig なし）
response = bedrock_client.converse(
    modelId=model_id,
    messages=messages,
    system=system_prompts
)

# レスポンスの表示
output_message = response['output']['message']
print(f"Role: {output_message['role']}")
print(f"\n{output_message['content'][0]['text']}")

# トークン使用量の表示
usage = response['usage']
print(f"\n--- トークン使用量 ---")
print(f"入力トークン数: {usage['inputTokens']}")
print(f"出力トークン数: {usage['outputTokens']}")
print(f"合計トークン数: {usage['totalTokens']}")

## 3-1. inferenceConfig を適用した Converse API の呼び出し

inferenceConfig を使用して、生成パラメータ（temperature、maxTokens、topP）を制御するサンプルです。  
system プロンプトと inferenceConfig の両方を使用しています。

### inferenceConfig を指定しない場合のデフォルト値（Amazon Nova Lite）

| パラメータ | デフォルト値 | 説明 |
|-----------|------------|------|
| temperature | 0.7 | ランダム性を制御（範囲: 0.00001〜1.0）。低いほど決定的な出力になる |
| topP | 0.9 | 核サンプリングの閾値（範囲: 0〜1.0）。低いほど集中した出力になる |
| maxTokens | リクエストコンテキストに基づく動的デフォルト（最大 5,000） | 生成される最大トークン数 |
| stopSequences | なし | 生成を停止する文字列のリスト |

※ temperature と topP は同時に調整せず、どちらか一方を調整することが推奨されています。  
※ 参照: [Amazon Nova - Request and response schema](https://docs.aws.amazon.com/nova/latest/nova2-userguide/request-response-schema.html) / [Using the Converse API](https://docs.aws.amazon.com/nova/latest/nova2-userguide/using-converse-api.html)

In [ ]:
import boto3

# Bedrock Runtime クライアントの作成
bedrock_client = boto3.client('bedrock-runtime')

# 使用するモデルの指定
model_id = "amazon.nova-lite-v1:0"

# 推論パラメータの設定
inference_config = {
    "temperature": 0.7,     # 創造性の高い回答を生成
    "maxTokens": 4096,     # 最大出力トークン数
    "topP": 0.9            # 上位90%の確率の中からサンプリング
}

# system プロンプトの設定（日本料理の専門家として振る舞う）
system_prompts = [
    {
        "text": "あなたは日本料理と食文化の専門家です。料理の歴史、調理法、地域性、季節感、食材の選び方について深い知識を持っています。質問に対して、具体的なレシピや調理のコツも含めて詳細に回答してください。"
    }
]

# ユーザーメッセージの作成
messages = [
    {
        "role": "user",
        "content": [{"text": "日本各地の代表的なご当地ラーメンについて教えてください。札幌、喜多方、東京、横浜、京都、博多など、各地域のラーメンの特徴、スープのベース、麺の種類、代表的なトッピング、その土地ならではのこだわりを詳しく解説してください。"}]
    }
]

# Converse API の呼び出し（system プロンプト + inferenceConfig）
response = bedrock_client.converse(
    modelId=model_id,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

# レスポンスの表示
output_message = response['output']['message']
print(f"Role: {output_message['role']}")
print(f"\n{output_message['content'][0]['text']}")

# トークン使用量とストップ理由の表示
usage = response['usage']
print(f"\n--- トークン使用量 ---")
print(f"入力トークン数: {usage['inputTokens']}")
print(f"出力トークン数: {usage['outputTokens']}")
print(f"合計トークン数: {usage['totalTokens']}")
print(f"停止理由: {response['stopReason']}")

## 3-2. additionalModelRequestFields で topK を指定する例（Claude）

⚠️ **注意: このサンプルはラボ環境では使用できません。**

Converse API の inferenceConfig では topK を直接指定できません。  
topK のようなモデル固有のパラメータは `additionalModelRequestFields` を使って指定します。  
ここでは Claude モデルで topK を設定する例を示します。

### topK とは

topK は、次のトークンを選択する際に確率の高い上位 K 個の候補のみを考慮するパラメータです。  
値を小さくすると出力がより決定的になり、大きくすると多様性が増します。

In [ ]:
import boto3

# Bedrock Runtime クライアントの作成
bedrock_client = boto3.client('bedrock-runtime')

# 使用するモデルの指定（Claude 3.5 Sonnet v2）
model_id = "anthropic.claude-3-5-sonnet-20241022-v2:0"

# 推論パラメータの設定（Converse API 標準パラメータ）
inference_config = {
    "temperature": 0.7,
    "maxTokens": 4096
}

# モデル固有の追加パラメータ（topK は additionalModelRequestFields で指定）
additional_model_fields = {
    "top_k": 200  # 上位200個のトークンからサンプリング
}

# system プロンプトの設定（日本の建築の専門家として振る舞う）
system_prompts = [
    {
        "text": "あなたは日本の建築と都市計画の専門家です。伝統建築から現代建築まで幅広い知識を持ち、建築様式の歴史的変遷、構造的特徴、文化的背景について詳しく解説できます。具体的な建築物の例を挙げながら回答してください。"
    }
]

# ユーザーメッセージの作成
messages = [
    {
        "role": "user",
        "content": [{"text": "日本の城の建築様式について詳しく教えてください。天守閣の構造、石垣の積み方の種類（野面積み、打込み接ぎ、切込み接ぎなど）、狭間や石落としなどの防御設備、そして姫路城・松本城・熊本城など代表的な名城の建築的特徴を比較しながら解説してください。"}]
    }
]

# Converse API の呼び出し（additionalModelRequestFields で topK を指定）
response = bedrock_client.converse(
    modelId=model_id,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config,
    additionalModelRequestFields=additional_model_fields
)

# レスポンスの表示
output_message = response['output']['message']
print(f"Role: {output_message['role']}")
print(f"\n{output_message['content'][0]['text']}")

# トークン使用量とストップ理由の表示
usage = response['usage']
print(f"\n--- トークン使用量 ---")
print(f"入力トークン数: {usage['inputTokens']}")
print(f"出力トークン数: {usage['outputTokens']}")
print(f"合計トークン数: {usage['totalTokens']}")
print(f"停止理由: {response['stopReason']}")

## 4. Converse API のストリーミングレスポンス

converse_stream を使用して、レスポンスをストリーミングで受け取るサンプルです。  
リアルタイムでテキストが生成される様子を確認できます。

In [ ]:
import boto3

def print_stream(stream):
    """ストリーミングレスポンスを処理して表示する関数"""
    if stream:
        for event in stream:
            # メッセージ開始イベント
            if 'messageStart' in event:
                print(f"\nRole: {event['messageStart']['role']}")

            # コンテンツブロックのデルタ（テキストの断片）
            if 'contentBlockDelta' in event:
                print(event['contentBlockDelta']['delta']['text'], end="")

            # メッセージ終了イベント
            if 'messageStop' in event:
                print(f"\n\n停止理由: {event['messageStop']['stopReason']}")

            # メタデータ（トークン使用量、レイテンシ）
            if 'metadata' in event:
                metadata = event['metadata']
                if 'usage' in metadata:
                    print(f"\n--- トークン使用量 ---")
                    print(f"入力トークン数: {metadata['usage']['inputTokens']}")
                    print(f"出力トークン数: {metadata['usage']['outputTokens']}")
                    print(f"合計トークン数: {metadata['usage']['totalTokens']}")
                if 'metrics' in metadata:
                    print(f"レイテンシ: {metadata['metrics']['latencyMs']} ミリ秒")


# Bedrock Runtime クライアントの作成
bedrock_client = boto3.client('bedrock-runtime')

# 使用するモデルの指定
model_id = "amazon.nova-lite-v1:0"

# 推論パラメータの設定
inference_config = {
    "temperature": 0.7,
    "maxTokens": 4096
}

# system プロンプトの設定（旅行ガイドとして振る舞う）
system_prompts = [
    {
        "text": "あなたは日本の観光地に詳しい旅行ガイドです。各地の名所、グルメ、温泉、歴史的建造物、自然景観について、旅行者が実際に訪れたくなるような魅力的で詳細な情報を提供してください。アクセス方法やおすすめの季節なども含めてください。"
    }
]

# ユーザーメッセージの作成
messages = [
    {
        "role": "user",
        "content": [{"text": "京都の隠れた名所を10箇所紹介してください。有名な観光地ではなく、地元の人が愛する穴場スポットや、あまり知られていない寺社仏閣、季節限定の絶景ポイントなどを、それぞれの魅力とおすすめの訪問時期とともに詳しく教えてください。"}]
    }
]

# Converse Stream API の呼び出し
response = bedrock_client.converse_stream(
    modelId=model_id,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

# ストリーミングレスポンスの処理
stream = response.get('stream')
print_stream(stream)